# De la exploración al reporte de mediciones

La primera notebook nos permitió comprobar formas, seleccionar rondas y resumir
columnas. Ahora usaremos esas operaciones como funciones reutilizables desde la
notebook y desde `main.py`, sin reconstruir la lógica en cada celda.

## Contenido

1. Cargar y validar la forma.
2. Seleccionar rondas completas.
3. Resumir por sala e interpretar el resultado.
4. Añadir una selección propia.

Abre esta notebook en VS Code desde `02-reporte-mediciones` y selecciona el kernel de `.venv`.


In [1]:
import json
from pathlib import Path

import numpy as np

from measurements.processing import (
    ROOM_NAMES,
    load_readings,
    select_complete_rows,
    summarize,
)

project_dir = Path.cwd()
data_dir = project_dir.parent / "datos"
if not (project_dir / "main.py").is_file() or not data_dir.is_dir():
    raise FileNotFoundError(
        "Start the notebook from the 02-reporte-mediciones project directory"
    )

## 1. Cargar y validar la forma

`load_readings` comprueba el encabezado y lee una matriz con tres columnas.
La comprobación impide asignar un resumen a otra sala si cambia el orden del CSV.
Abre el módulo en el editor e identifica qué parte lee y cuál valida.

[Lectura con loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [2]:
readings = load_readings(data_dir / "readings.csv")
print(ROOM_NAMES)
print(readings.shape)
print(readings[:2])
assert readings.shape == (8, 3)

('north_room', 'south_room', 'storage_room')
(8, 3)
[[22. 24. 20.]
 [23. 25. 21.]]


## 2. Seleccionar rondas completas

La función devuelve dos resultados: las filas seleccionadas y la cantidad de
rechazos. Esto permite conservar evidencia del filtrado. Ni el arreglo original
ni el CSV se modifican.

In [3]:
quality_readings = load_readings(data_dir / "readings_quality.csv")
original = quality_readings.copy()
complete, rejected = select_complete_rows(quality_readings)
print(complete)
print(f"Rejected rows: {rejected}")
assert rejected == 2
assert not np.shares_memory(complete, quality_readings)
np.testing.assert_array_equal(quality_readings, original)

[[22. 24. 20.]
 [24. 26. 22.]]
Rejected rows: 2


### Comprobación: ninguna ronda completa

No calcularemos medias sobre una matriz vacía. La función comunica el problema
con una excepción; la notebook puede capturarla para discutir el caso y continuar.

In [4]:
invalid_readings = np.array([[np.nan, 24, 20], [22, np.inf, 21]])
try:
    select_complete_rows(invalid_readings)
except ValueError as error:
    print(f"Expected error: {error}")
else:
    raise AssertionError("Expected an error for data without complete rows")

Expected error: No complete readings remain


## 3. Resumir por sala

Las operaciones numéricas se aplican por eje. El pequeño ciclo de la función
`summarize` asigna nombres a los tres resultados y convierte los escalares a
`float` de Python para serializarlos en JSON. No recorre las mediciones para
calcular sus medias.

In [5]:
report = summarize(complete)
print(json.dumps(report, indent=2, allow_nan=False))
np.testing.assert_allclose(
    [report[name]["mean_c"] for name in ROOM_NAMES], [23, 25, 21]
)

{
  "north_room": {
    "mean_c": 23.0,
    "min_c": 22.0,
    "max_c": 24.0
  },
  "south_room": {
    "mean_c": 25.0,
    "min_c": 24.0,
    "max_c": 26.0
  },
  "storage_room": {
    "mean_c": 21.0,
    "min_c": 20.0,
    "max_c": 22.0
  }
}


### Ejercicio 10: explicar el cambio de medias

Compara el reporte de las ocho rondas originales con el de las dos rondas
completas del archivo con errores. Describe qué datos entraron en cada cálculo.
No son dos momentos de una instalación real: son archivos de práctica distintos.

In [6]:
# Compare the summaries and write your interpretation.
original_report = summarize(load_readings(data_dir / "readings.csv"))
quality_report = summarize(complete)
print("Original report:")
print(json.dumps(original_report, indent=2, allow_nan=False))
print("\nComplete rows report:")
print(json.dumps(quality_report, indent=2, allow_nan=False))

# Las ocho rondas del archivo original incluyen todas las mediciones, por eso
# sus medias son 25.5, 27.5 y 23.5. El archivo con errores solo usa dos
# rondas completas, así que el promedio se calcula sobre un subconjunto
# distinto: 22, 24, 20 y 24, 26, 22. No son dos momentos del mismo sistema,
# sino dos archivos de práctica distintos con distintos datos de entrada.


Original report:
{
  "north_room": {
    "mean_c": 25.5,
    "min_c": 22.0,
    "max_c": 29.0
  },
  "south_room": {
    "mean_c": 27.5,
    "min_c": 24.0,
    "max_c": 31.0
  },
  "storage_room": {
    "mean_c": 23.5,
    "min_c": 20.0,
    "max_c": 27.0
  }
}

Complete rows report:
{
  "north_room": {
    "mean_c": 23.0,
    "min_c": 22.0,
    "max_c": 24.0
  },
  "south_room": {
    "mean_c": 25.0,
    "min_c": 24.0,
    "max_c": 26.0
  },
  "storage_room": {
    "mean_c": 21.0,
    "min_c": 20.0,
    "max_c": 22.0
  }
}


## 4. Añadir una selección propia

### Ejercicio 11: seleccionar rondas cálidas

Resuelve la [práctica](../PRACTICA.md) creando `select_warm_rounds` en un módulo
nuevo. Reinicia el kernel después de guardar el módulo y comprueba aquí las filas
seleccionadas mediante `assert`.

Para umbral `27`, deben quedar tres rondas del archivo original. Para umbral
`100`, la selección es vacía y no se debe calcular su media.

In [7]:
from measurements.selection import select_warm_rounds

warm_rounds = select_warm_rounds(readings, 27)
print(warm_rounds)
assert np.array_equal(
    warm_rounds,
    np.array([[27, 29, 25], [28, 30, 26], [29, 31, 27]]),
)
assert warm_rounds.shape == (3, 3)

empty_selection = select_warm_rounds(readings, 100)
print(empty_selection)
assert empty_selection.shape == (0, 3)
# La selección vacía es válida: significa que ninguna fila cumple la
# condición. Como no hay filas, no tiene sentido calcular una media.

invalid = np.array([[np.nan, 24, 20], [22, 23, 21]])
try:
    select_warm_rounds(invalid, 27)
except ValueError as error:
    print(f"Expected error: {error}")
else:
    raise AssertionError("Expected a ValueError for non-finite readings")

modifiable = select_warm_rounds(readings, 24)
modifiable[0, 0] = 99
assert readings[0, 0] == 22

np.testing.assert_allclose(warm_rounds.mean(axis=0), [28.0, 30.0, 26.0])
np.testing.assert_allclose(readings.mean(axis=0), [25.5, 27.5, 23.5])
# En la selección cálida solo entran las filas 27, 28 y 29 de la sala norte.
# En el conjunto completo entran todas las ocho rondas.


[[27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]
[]
Expected error: Readings must be finite


## Verificación final

Desde la terminal del proyecto ejecuta `uv run --locked python main.py` para
generar los archivos. Después reinicia el kernel y ejecuta toda la notebook.
Las pruebas existentes con pytest son una comprobación opcional.
